# 03 · Entrenamiento (MLflow) + Model Registry
### Prueba final MLOps · Cesar Romero

**Qué exige el examen en esta etapa**

*Punto 3 — Entrenamiento*
- Entrenar y comparar **mínimo 5 variantes** de modelos o configuraciones.
- Registrar cada ejecución con **MLflow**.
- Experimento: **`control-2`**
- Runs: **`modelo-run-x`** (x = número de ejecución)

*Punto 4 — Registro y selección*
- Seleccionar el mejor modelo con una métrica de evaluación.
- Registrarlo en **MLflow Model Registry** con el alias **`PRINCIPAL`**.

**Entrada:** `cancer_raw` (labels) + `cancer_features` (Feature Store)
**Salida:** modelo `<catalog>.mlops_final.cancer_classifier` con alias `PRINCIPAL`

In [ ]:
%pip install databricks-feature-engineering --quiet
%restart_python

## 1. Configuración

Dos nombres críticos para la corrección del examen:

- `EXPERIMENT_NAME = "/Shared/control-2"` → en la UI de MLflow aparece como **control-2**.
  Va en `/Shared/` para que el Job (que corre con otra identidad) pueda escribir en él.
- `MODEL_NAME` es un nombre de **tres niveles** (`catalog.schema.model`), obligatorio
  cuando el registry es Unity Catalog.

In [ ]:
import mlflow, pandas as pd, numpy as np
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
SCHEMA  = "mlops_final"

RAW_TABLE     = f"{CATALOG}.{SCHEMA}.cancer_raw"
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.cancer_features"

EXPERIMENT_NAME = "/Shared/control-2"        # ← nombre exigido: control-2
MODEL_NAME      = f"{CATALOG}.{SCHEMA}.cancer_classifier"
ALIAS           = "PRINCIPAL"                # ← alias exigido

mlflow.set_registry_uri("databricks-uc")     # registry = Unity Catalog
mlflow.set_experiment(EXPERIMENT_NAME)

fe     = FeatureEngineeringClient()
client = MlflowClient()

print("Experimento :", EXPERIMENT_NAME)
print("Modelo UC   :", MODEL_NAME)
print("Alias       :", ALIAS)

## 2. Training set desde Feature Store

Este es el corazón conceptual de Feature Store y lo que diferencia una entrega buena de
una mediocre: **no leemos la tabla de features directamente**.

Partimos sólo de `patient_id` + `target`, y `FeatureLookup` resuelve el join contra el
Feature Store usando la clave. Ventaja: MLflow queda con el **lineage** registrado —
sabe exactamente qué Feature Table y qué columnas alimentaron el modelo.

`exclude_columns=["patient_id"]` saca el ID del set de entrenamiento: es un identificador,
no una señal predictiva. Si lo dejáramos, el modelo aprendería del orden de las filas.

In [ ]:
labels_df = spark.table(RAW_TABLE).select("patient_id", "target")

feature_lookups = [
    FeatureLookup(table_name=FEATURE_TABLE, lookup_key="patient_id")
]

training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=feature_lookups,
    label="target",
    exclude_columns=["patient_id"],
)

training_sdf = training_set.load_df()
display(training_sdf.limit(10))

## 3. Versiones de datos (auditoría)

Guardamos la versión Delta de cada tabla en el momento del entrenamiento. Se registran
como **params** de cada run: si mañana alguien pregunta "¿con qué datos se entrenó esto?",
la respuesta es reproducible vía Delta Time Travel (`VERSION AS OF`).

In [ ]:
def delta_version(table: str) -> int:
    return (spark.sql(f"DESCRIBE HISTORY {table}")
                 .selectExpr("max(version) as v").collect()[0]["v"])

RAW_VERSION     = delta_version(RAW_TABLE)
FEATURE_VERSION = delta_version(FEATURE_TABLE)

print(f"{RAW_TABLE} → versión {RAW_VERSION}")
print(f"{FEATURE_TABLE} → versión {FEATURE_VERSION}")

## 4. Split reproducible

El dataset es pequeño (569 filas) → pandas es suficiente y más rápido que Spark aquí.

- `random_state=42` → el split es idéntico en cada ejecución del Job.
- `stratify=y` → conserva la proporción benigno/maligno en train y test. Sin esto,
  el AUC entre modelos no sería comparable porque cada uno vería una distribución distinta.

**Los 5 modelos comparten exactamente el mismo split.** Es la única forma de que la
comparación sea honesta.

In [ ]:
pdf = training_sdf.toPandas()

X = pdf.drop(columns=["target"])
y = pdf["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features ({X.shape[1]}): {list(X.columns)}")
print(f"Proporción benigno — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

## 5. Las 5 variantes

| Run | Modelo | Por qué está aquí |
|---|---|---|
| `modelo-run-1` | Logistic Regression + scaler | Baseline lineal interpretable. Sensible a escala → necesita el scaler. |
| `modelo-run-2` | Random Forest (100, depth 5) | Bagging. Inmune a la escala, robusto a outliers. |
| `modelo-run-3` | Gradient Boosting (100, lr 0.1) | Boosting secuencial: corrige errores del árbol anterior. Suele ganar en tabular. |
| `modelo-run-4` | SVM RBF + scaler | Frontera no lineal vía kernel. `probability=True` para poder calcular AUC. |
| `modelo-run-5` | KNN (k=7) + scaler | Sin fase de entrenamiento real; depende 100% de la distancia → el scaler es crítico. |

**Todos van dentro de un `Pipeline`.** Esto no es cosmético: el `StandardScaler` queda
*dentro* del artefacto del modelo, así que cuando el endpoint REST reciba datos crudos
en el paso 5, el escalado se aplica solo. Si el scaler viviera fuera, el endpoint
devolvería basura.

In [ ]:
VARIANTES = [
    ("modelo-run-1", "logistic_regression",
     Pipeline([("scaler", StandardScaler()),
               ("clf", LogisticRegression(max_iter=1000, random_state=42))])),

    ("modelo-run-2", "random_forest",
     Pipeline([("clf", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))])),

    ("modelo-run-3", "gradient_boosting",
     Pipeline([("clf", GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                  max_depth=3, random_state=42))])),

    ("modelo-run-4", "svm_rbf",
     Pipeline([("scaler", StandardScaler()),
               ("clf", SVC(kernel="rbf", C=1.0, probability=True, random_state=42))])),

    ("modelo-run-5", "knn",
     Pipeline([("scaler", StandardScaler()),
               ("clf", KNeighborsClassifier(n_neighbors=7))])),
]

for run_name, algo, _ in VARIANTES:
    print(f"{run_name:15s} → {algo}")

## 6. Entrenamiento con tracking MLflow

Una función, cinco llamadas. Cada run registra las cuatro piezas que MLflow considera
una ejecución completa:

1. **Params** — hiperparámetros + nombres y versiones de las tablas de origen.
2. **Metrics** — accuracy, precision, recall, F1, **AUC** (métrica de selección).
3. **Model** — el `Pipeline` completo, serializado.
4. **Signature + input_example** — el contrato de entrada. Sin esto, Model Serving no
   valida el esquema y el endpoint acepta payloads malformados silenciosamente.

In [ ]:
def entrenar_run(run_name, algoritmo, pipeline):
    with mlflow.start_run(run_name=run_name) as run:

        pipeline.fit(X_train, y_train)

        y_pred  = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy":  accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall":    recall_score(y_test, y_pred),
            "f1":        f1_score(y_test, y_pred),
            "auc":       roc_auc_score(y_test, y_proba),
        }

        # 1) Parámetros: hiperparámetros reales + trazabilidad de datos
        mlflow.log_param("algoritmo", algoritmo)
        for k, v in pipeline.named_steps["clf"].get_params().items():
            mlflow.log_param(f"clf_{k}", v)
        mlflow.log_param("raw_table",       RAW_TABLE)
        mlflow.log_param("raw_version",     RAW_VERSION)
        mlflow.log_param("feature_table",   FEATURE_TABLE)
        mlflow.log_param("feature_version", FEATURE_VERSION)
        mlflow.log_param("n_features",      X.shape[1])
        mlflow.log_param("test_size",       0.2)

        # 2) Métricas
        mlflow.log_metrics(metrics)

        # 3+4) Modelo con firma e input example
        signature = infer_signature(X_train, pipeline.predict(X_train))
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model",
            signature=signature,
            input_example=X_train.head(3),
        )

        mlflow.set_tag("variante", run_name)
        print(f"✅ {run_name:15s} AUC={metrics['auc']:.4f}  F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}")

        return {"run_id": run.info.run_id, "run_name": run_name,
                "algoritmo": algoritmo, **metrics}


resultados = [entrenar_run(rn, algo, pipe) for rn, algo, pipe in VARIANTES]

tabla = pd.DataFrame(resultados).sort_values("auc", ascending=False).reset_index(drop=True)
display(tabla)

## 7. Comparación desde el tracking server

Reconsultamos el experimento con `search_runs` en vez de usar la lista en memoria.
Esto verifica que los runs quedaron realmente persistidos en `control-2` — que es lo
que el evaluador verá en el screenshot.

In [ ]:
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
assert exp is not None, f"No existe el experimento {EXPERIMENT_NAME}"

runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.auc DESC"],
    max_results=10,
)

cols = ["run_id", "tags.mlflow.runName", "params.algoritmo",
        "metrics.accuracy", "metrics.f1", "metrics.auc",
        "params.raw_version", "params.feature_version"]
display(runs[[c for c in cols if c in runs.columns]])

print(f"Runs registrados en '{EXPERIMENT_NAME}': {len(runs)}")

---
# Punto 4 · Registro y selección del mejor modelo

## 8. Criterio de selección: AUC

**Por qué AUC y no accuracy:** el dataset está desbalanceado (≈63% benigno). Un modelo
que prediga "benigno" siempre saca 63% de accuracy sin aprender nada. AUC mide la
capacidad de **ranking** — qué tan bien separa las dos clases en todos los umbrales
posibles — y es insensible a ese desbalance.

En un caso clínico real probablemente priorizaríamos **recall** sobre la clase maligna
(un falso negativo es un cáncer no detectado). Dejo AUC porque es la métrica que el
examen espera para comparar familias de modelos, pero vale la pena mencionarlo si te
preguntan en la defensa.

In [ ]:
best = runs.iloc[0]
best_run_id   = best["run_id"]
best_run_name = best["tags.mlflow.runName"]

print(f"🏆 Mejor run : {best_run_name}")
print(f"   run_id    : {best_run_id}")
print(f"   algoritmo : {best.get('params.algoritmo')}")
print(f"   AUC       : {best['metrics.auc']:.4f}")
print(f"   F1        : {best['metrics.f1']:.4f}")
print(f"   Accuracy  : {best['metrics.accuracy']:.4f}")

## 9. Registro en Model Registry (Unity Catalog)

`runs:/<run_id>/model` apunta al artefacto guardado en el paso 6. `register_model`
crea una **nueva versión** del modelo gobernado — no sobrescribe las anteriores.

In [ ]:
mv = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name=MODEL_NAME,
)

print(f"✅ Modelo registrado: {MODEL_NAME}")
print(f"   Versión: {mv.version}")

## 10. Alias `PRINCIPAL`

El alias es un puntero móvil. Los consumidores (el endpoint del paso 5, un job de batch
inference) referencian `models:/catalog.schema.cancer_classifier@PRINCIPAL` y nunca un
número de versión. Promover un modelo nuevo = mover el alias. Rollback = moverlo de vuelta.
Ningún consumidor cambia una línea de código.

In [ ]:
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias=ALIAS,
    version=mv.version,
)

check = client.get_model_version_by_alias(name=MODEL_NAME, alias=ALIAS)
print(f"✅ Alias '{ALIAS}' → {MODEL_NAME} versión {check.version}")

client.update_model_version(
    name=MODEL_NAME,
    version=mv.version,
    description=(
        f"Mejor modelo de control-2. Run: {best_run_name} ({best.get('params.algoritmo')}). "
        f"AUC={best['metrics.auc']:.4f}. "
        f"Datos: {RAW_TABLE} v{RAW_VERSION}, {FEATURE_TABLE} v{FEATURE_VERSION}."
    ),
)
print("Descripción de versión actualizada.")

## 11. Smoke test

Cargamos el modelo **por alias** (no por versión) y predecimos una fila. Si esto funciona,
el endpoint del paso 5 también funcionará: usa exactamente el mismo artefacto y la misma firma.

In [ ]:
modelo = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{ALIAS}")

ejemplo = pd.DataFrame([{
    "radius": 14.0, "texture": 20.0, "perimeter": 90.0, "area": 600.0,
    "smoothness": 0.10, "compactness": 0.12,
    "area_radius_ratio":      600.0 / (14.0 ** 2),
    "perimeter_radius_ratio": 90.0 / 14.0,
    "compactness_x_texture":  0.12 * 20.0,
}])[list(X.columns)]          # mismo orden de columnas que la firma

pred = modelo.predict(ejemplo)
ejemplo["prediction"] = pred
print("Predicción:", pred, " (1 = benigno, 0 = maligno)")
display(ejemplo)

print("\n📋 Payload listo para Postman (paso 5):")
import json
print(json.dumps({"dataframe_records": ejemplo.drop(columns=["prediction"]).to_dict(orient="records")}, indent=2))

---
### Cierre notebook 03

```
Feature Store + labels
   → training set (lineage registrado)
   → 5 runs en el experimento control-2
   → selección por AUC
   → cancer_classifier vN en Unity Catalog
   → alias PRINCIPAL
```

**Evidencias para el examen:**
- *Punto 3*: screenshot de **Experiments → control-2** con los 5 runs `modelo-run-1..5`
  y sus métricas (activa las columnas `auc`, `f1`, `accuracy`).
- *Punto 4*: screenshot de **Catalog Explorer → cancer_classifier** mostrando la versión
  y el alias `PRINCIPAL`.

Sigue con el **paso 5** (Model Serving + Postman) usando el payload impreso arriba.